## Data Preparation — African Crop Yield Intelligence

Ce notebook applique les corrections identifiées lors du Data Understanding et produit les jeux de données préparés. Il fait suite à `02_data_understanding.ipynb` et à la décision de passer en Data Preparation sous trois conditions.

**Périmètre**

- Les fichiers `regional`, `survey` et `cropcut` sont préparés séparément, conformément à la première condition de la décision.
- Le fichier `point` est exclu, car il ne porte aucune des cibles retenues.

**Cibles retenues**

- `yieldTonHa` de `regional` aux niveaux administratifs 1 et 2, pour la généralisation spatiale et temporelle.
- `YieldUsingGPS_ton_ha_` de `survey`, à l'échelle de la parcelle.
- `Yield_ton_ha_` de `cropcut`, comme référence de validation pour l'Éthiopie.

**Méthode**

Toutes les corrections sont implémentées sous forme de fonctions testées dans `src/data/`, ce qui satisfait la deuxième condition de la décision. Ce notebook se limite à les exécuter, à contrôler leurs effets et à exporter les résultats. Les fichiers bruts ne sont jamais modifiés. Chaque correction ajoute un indicateur préfixé par `flag_`, de sorte que toute ligne modifiée ou écartée reste traçable.

In [1]:
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.data.load import DOSSIER_TRAITE, charger_brut
from src.data.prepare import exporter, preparer_tout

pd.set_option("display.max_columns", 40)

### 1. Tests unitaires

La suite de tests vérifie chaque fonction de préparation sur des cas construits pour l'occasion avant toute exécution sur les données réelles.

In [2]:
execution = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-t", "."],
    cwd=RACINE, capture_output=True, text=True,
)
print(execution.stderr.strip().splitlines()[-3:])

['Ran 25 tests in 0.345s', '', 'OK']


### 2. Exécution du pipeline

Les fichiers bruts sont chargés depuis le cache de `data/interim`, indexé par l'empreinte SHA-256 de chaque fichier source. Une copie de chaque tableau brut est conservée afin de vérifier en fin de notebook que la préparation ne l'a pas modifié.

In [3]:
bruts = {nom: charger_brut(nom) for nom in ["regional", "survey", "cropcut"]}
copies_brutes = {nom: df.copy() for nom, df in bruts.items()}
tableaux = preparer_tout(bruts)

regional = tableaux["regional_prepare"]
survey = tableaux["survey_prepare"]
cropcut = tableaux["cropcut_prepare"]


def bilan_indicateurs(df):
    colonnes = [colonne for colonne in df.columns if colonne.startswith("flag_")]
    return df[colonnes].sum().rename("lignes_signalees").to_frame()

### 3. Préparation de `regional`

Les corrections appliquées sont, dans l'ordre, la normalisation des textes, le remplacement des codes -999 et -99, la permutation de la surface et de la production sur les lignes inversées, l'unification des codes du Ghana et des libellés de pays, l'harmonisation de la saison principale éthiopienne, la reconstruction du niveau administratif, puis la sélection d'une ligne par cellule.

In [4]:
brut = bruts["regional"]
print("Lignes :", len(brut), "->", len(regional))
print("Rendements manquants :", brut["yieldTonHa"].isna().sum(), "->", regional["yieldTonHa"].isna().sum())
print("Rendements négatifs :", (brut["yieldTonHa"] < 0).sum(), "->", (regional["yieldTonHa"] < 0).sum())
print("Codes pays L0_GID :", brut["L0_GID"].nunique(), "->", regional["L0_GID"].nunique())
print("Libellés de pays :", brut["country"].nunique(), "->", regional["country"].nunique())
print("Niveaux manquants :", brut["level"].isna().sum(), "->", regional["level"].isna().sum())
display(bilan_indicateurs(regional))

Lignes : 197535 -> 197535
Rendements manquants : 34584 -> 34650
Rendements négatifs : 66 -> 0
Codes pays L0_GID : 71 -> 47
Libellés de pays : 49 -> 47
Niveaux manquants : 491 -> 6


,lignes_signalees
flag_surface_production_permutees,33740
flag_relation_rendement_inexpliquee,1710
flag_niveau_ambigu,6
flag_gid_incomplet,36
flag_hierarchie_gid_incoherente,488
flag_rendement_hors_bornes,4657
flag_doublon_cellule,182


In [5]:
print("Lignes écartées comme doublons de cellule, par source :")
print(regional.loc[regional["flag_doublon_cellule"], "Source"].value_counts().to_dict())
print("\nRendements hors bornes, par source :")
print(regional.loc[regional["flag_rendement_hors_bornes"], "Source"].value_counts().to_dict())

modelisation = tableaux["regional_modelisation"]
print("\nJeu de modélisation :", len(modelisation), "lignes")
print("Niveaux :", modelisation["level"].value_counts().sort_index().to_dict())
pays_absents = sorted(set(regional["country"]) - set(modelisation["country"]))
print("Pays présents uniquement au niveau national :", pays_absents)
print("Pays :", modelisation["L0_GID"].nunique(), "| cultures :", modelisation["crop"].nunique(),
      "| années :", modelisation["harvest_year"].min(), "-", modelisation["harvest_year"].max())
print("Sources :", modelisation["Source"].value_counts().to_dict())

Lignes écartées comme doublons de cellule, par source :
{'FAO_subnational': 105, 'HarvestStat': 73, 'RCeA': 4}

Rendements hors bornes, par source :
{'HarvestStat': 1901, 'FAOSTAT': 1784, 'FAO_subnational': 382, 'RCeA': 357, 'AgroMAPS': 231, 'USAID_KadunaState': 2}

Jeu de modélisation : 146151 lignes
Niveaux : {np.int64(1): 57677, np.int64(2): 88474}


Pays présents uniquement au niveau national : ['Djibouti', 'Equatorial Guinea']
Pays : 45 | cultures : 34 | années : 1960 - 2023
Sources : {'HarvestStat': 118715, 'RCeA': 10711, 'AgroMAPS': 9958, 'FAO_subnational': 6695, 'USAID_KadunaState': 72}


**Constat**

- Les 66 rendements négatifs ont disparu. Les rendements manquants passent de 34 584 à 34 650, les 36 codes -99 et les 30 valeurs négatives dérivées du code -999 étant désormais traités comme manquants.
- Les 33 740 lignes inversées ont été permutées. Les 1 710 lignes qui ne vérifient aucune relation entre production, surface et rendement restent signalées.
- Les 71 codes `L0_GID` deviennent 47 et les 49 libellés de pays deviennent 47, chaque code ne portant plus qu'un nom. Le niveau administratif est reconstruit pour 485 des 491 lignes, les 6 lignes ambiguës restant signalées.
- La règle de sélection écarte 182 lignes, dont 105 lignes `FAO_subnational` et 4 lignes `RCeA` qui recouvraient des lignes `HarvestStat`. Ce recouvrement n'est apparu qu'après harmonisation des saisons, `HarvestStat` nommant Meher la saison que les autres sources nomment Main en Éthiopie.
- 4 657 rendements sortent des bornes de plausibilité de leur culture, dont 1 784 lignes `FAOSTAT` de niveau national.
- Le jeu de modélisation compte 146 151 lignes réparties sur 45 pays, 34 cultures et la période 1960 à 2023. Djibouti et la Guinée équatoriale en sont absents, car ils ne disposent que de données nationales.

**Analyse**

La hiérarchie GADM reste incohérente pour 488 lignes ghanéennes, dont l'identifiant de niveau 2 est rattaché à plusieurs régions. Seuls les polygones GADM permettraient de les corriger. `HarvestStat` représente 81 % du jeu de modélisation, ce qui en fait la source dominante de la cible.

**Conséquences pour la suite**

- Les lignes ghanéennes signalées par `flag_hierarchie_gid_incoherente` devront être exclues de toute agrégation au niveau 1 tant que leur rattachement n'est pas corrigé.
- La source devra être prise en compte dans l'évaluation, compte tenu de la prépondérance de `HarvestStat`.

### 4. Préparation de `survey`

Les corrections appliquées sont la normalisation des textes, la suppression des copies exactes, l'harmonisation du libellé de la Côte d'Ivoire et de l'année de récolte, le signalement des valeurs impossibles et la construction des identifiants de ligne, de ménage et de grappe.

In [6]:
brut = bruts["survey"]
print("Lignes :", len(brut), "->", len(survey))
print("Pays :", sorted(set(brut["Country"]) - set(survey["Country"])), "->", sorted(set(survey["Country"]) - set(brut["Country"])))
print("AgYearEnd manquant :", brut["AgYearEnd"].isna().sum(), "->", survey["AgYearEnd"].isna().sum())
display(bilan_indicateurs(survey))

modelisation = tableaux["survey_modelisation"]
print("Jeu de modélisation :", len(modelisation), "lignes")
print("Identifiant de ménage manquant :", modelisation["id_menage"].isna().sum(),
      "| identifiant de grappe manquant :", modelisation["id_grappe"].isna().sum())
print("\nLignes par pays, rendement fondé sur la surface déclarée et sur la surface GPS :")
display(survey.groupby("Country")[["Yield_ton_ha_", "YieldUsingGPS_ton_ha_"]].count())
print("Pays :", modelisation["Country"].nunique(), "| cultures :", modelisation["Crop"].nunique(),
      "| années :", modelisation["annee_recolte"].min(), "-", modelisation["annee_recolte"].max())

Lignes : 195152 -> 194704


Pays : ['Cote dIvoire'] -> ['Côte dIvoire']
AgYearEnd manquant : 218 -> 0


,lignes_signalees
flag_prccrop_invalide,80
flag_surface_aberrante,1
flag_hhid_tronque,3416
flag_hierarchie_gid_incoherente,10
flag_rendement_hors_bornes,737


Jeu de modélisation : 151138 lignes
Identifiant de ménage manquant : 6106 | identifiant de grappe manquant : 13423

Lignes par pays, rendement fondé sur la surface déclarée et sur la surface GPS :


,Yield_ton_ha_,YieldUsingGPS_ton_ha_
Country,,
Benin,3834,69
Burkina Faso,1570,0
Côte dIvoire,1333,609
Ethiopia,18682,27504
Guinea-Bissau,1368,1071
Malawi,49137,48352
Mali,4037,5643
Niger,8862,13561
Nigeria,4638,11545


Pays : 12 | cultures : 25 | années : 2005 - 2022


**Constat**

- Les 448 copies exactes sont supprimées et le fichier passe de 195 152 à 194 704 lignes. L'année de fin de campagne est désormais renseignée pour toutes les lignes.
- 3 416 identifiants de ménage tronqués sont signalés. L'identifiant de ménage est manquant pour 6 106 lignes du jeu de modélisation et l'identifiant de grappe pour 13 423 lignes, faute de coordonnées.
- Le jeu de modélisation compte 151 138 lignes, 25 cultures et la période 2005 à 2022, mais seulement 12 pays sur 13.
- La cible fondée sur la surface GPS est absente pour l'ensemble du Burkina Faso et quasi absente pour le Bénin, avec 69 lignes contre 3 834 pour la cible fondée sur la surface déclarée. Elle est également réduite de moitié pour la Côte d'Ivoire.

**Analyse**

Les enquêtes du Burkina Faso et, en grande partie, celles du Bénin n'ont pas mesuré la surface par GPS. Le choix de `YieldUsingGPS_ton_ha_` comme cible écarte donc ces pays du jeu de modélisation. Ce point n'avait pas été mis en évidence pays par pays lors du Data Understanding. À l'inverse, l'Éthiopie, le Mali, le Niger et le Nigeria comptent davantage de rendements GPS que de rendements fondés sur la surface déclarée.

**Conséquences pour la suite**

- Le choix de la cible de `survey` doit être confirmé au regard de cette perte de couverture. Une alternative consisterait à recourir à `Yield_ton_ha_` lorsque la surface GPS est absente, en conservant un indicateur de méthode de mesure.
- Lorsque l'identifiant de ménage est manquant, les partitions devront s'appuyer sur la grappe ou, à défaut, sur `L2_GID`.

### 5. Préparation de `cropcut`

Les corrections appliquées sont la normalisation des textes, la reconstruction du ménage par division entière de `holder_id` par 100, le signalement des poids incohérents et des dates impossibles dans le calendrier éthiopien, et la construction de l'année de récolte et de la grappe.

In [7]:
display(bilan_indicateurs(cropcut))
validation = tableaux["cropcut_validation"]
print("Jeu de validation :", len(validation), "lignes")
print("Ménages reconstruits :", cropcut["id_menage"].nunique(), "| household_id d'origine :", bruts["cropcut"]["household_id"].nunique())
print("Identifiant de ménage manquant, par vague :", cropcut.loc[cropcut["id_menage"].isna(), "Source"].value_counts().to_dict())

,lignes_signalees
flag_poids_incoherent,80
flag_date_mesure_invalide,19
flag_rendement_hors_bornes,2


Jeu de validation : 11263 lignes
Ménages reconstruits : 2338 | household_id d'origine : 2334
Identifiant de ménage manquant, par vague : {'ETH2018': 2905}


**Constat**

- 80 mesures présentent un poids sec supérieur au poids frais et 19 dates de mesure sont impossibles, même dans le calendrier éthiopien. Seuls 2 rendements sortent des bornes de plausibilité.
- La reconstruction produit 2 338 ménages, contre 2 334 valeurs d'origine de `household_id`. Les deux valeurs de substitution de `ETH2011` sont remplacées par les ménages réels qu'elles masquaient.
- L'identifiant de ménage reste manquant pour les 2 905 lignes de `ETH2018`, vague qui ne fournit pas `holder_id`.
- Le jeu de validation compte 11 263 lignes.

**Conséquences pour la suite**

Les dates invalides n'affectent pas la cible et sont seulement signalées. Les mesures de `ETH2018` devront être regroupées par grappe pour toute validation par groupe.

### 6. Contrôles d'intégrité

Ces contrôles vérifient les propriétés attendues des jeux produits. Le notebook s'interrompt si l'un d'eux échoue.

In [8]:
controles = {
    "aucun rendement négatif dans regional": (regional["yieldTonHa"].dropna() >= 0).all(),
    "un seul nom de pays par L0_GID": regional.groupby("L0_GID")["country"].nunique().max() == 1,
    "une ligne retenue par cellule dans regional": not tableaux["regional_modelisation"].duplicated(
        subset=["GDAM_ID", "L1_GID", "L2_GID", "crop", "saison", "harvest_year"]).any(),
    "identifiant de ligne unique dans survey": survey["id_ligne"].is_unique,
    "aucune copie exacte dans survey": not survey.drop(columns=[c for c in survey.columns if c.startswith(("flag_", "id_", "annee_"))]).duplicated().any(),
    "cibles renseignées dans les jeux finaux": all(
        tableaux[nom][cible].notna().all()
        for nom, cible in [("regional_modelisation", "yieldTonHa"), ("survey_modelisation", "YieldUsingGPS_ton_ha_"), ("cropcut_validation", "Yield_ton_ha_")]
    ),
    "données brutes non modifiées par la préparation": all(bruts[nom].equals(copies_brutes[nom]) for nom in bruts),
}
for nom, resultat in controles.items():
    print(f"{'OK ' if resultat else 'ÉCHEC'} {nom}")
assert all(controles.values())

OK  aucun rendement négatif dans regional
OK  un seul nom de pays par L0_GID
OK  une ligne retenue par cellule dans regional
OK  identifiant de ligne unique dans survey
OK  aucune copie exacte dans survey
OK  cibles renseignées dans les jeux finaux
OK  données brutes non modifiées par la préparation


### 7. Export des jeux préparés

Les tableaux sont écrits au format CSV compressé dans `data/processed`. Ce dossier est exclu du dépôt git, les fichiers étant entièrement reproductibles à partir des données brutes et du code.

In [9]:
exporter(tableaux)
for nom, tableau in tableaux.items():
    chemin = DOSSIER_TRAITE / f"{nom}.csv.gz"
    print(f"{nom:<24} {tableau.shape[0]:>7} lignes  {tableau.shape[1]:>3} colonnes  {chemin.stat().st_size / 1e6:6.2f} Mo")

bornes_plausibilite           34 lignes    2 colonnes    0.00 Mo
regional_prepare          197535 lignes   33 colonnes    4.04 Mo
regional_modelisation     146151 lignes   33 colonnes    3.45 Mo
survey_prepare            194704 lignes   36 colonnes    7.53 Mo
survey_modelisation       151138 lignes   36 colonnes    6.22 Mo
cropcut_prepare            11345 lignes   26 colonnes    0.26 Mo
cropcut_validation         11263 lignes   26 colonnes    0.25 Mo


### 8. Synthèse

| Jeu produit | Lignes | Usage |
|---|---:|---|
| `regional_prepare` | 197 535 | Fichier regional corrigé, avec tous ses indicateurs |
| `regional_modelisation` | 146 151 | Cible principale aux niveaux 1 et 2, une ligne par cellule |
| `survey_prepare` | 194 704 | Fichier survey corrigé, avec tous ses indicateurs |
| `survey_modelisation` | 151 138 | Cible à l'échelle de la parcelle |
| `cropcut_prepare` | 11 345 | Fichier cropcut corrigé, avec tous ses indicateurs |
| `cropcut_validation` | 11 263 | Référence de validation pour l'Éthiopie |
| `bornes_plausibilite` | 34 | Bornes du filtre des auteurs par culture |

**Décision**

Les deux premières conditions de la décision issue du Data Understanding sont remplies. Les fichiers sont traités séparément et les corrections sont implémentées sous forme de fonctions testées. La troisième condition, la définition du protocole d'évaluation par groupe et par période, reste à remplir avant l'entraînement du premier modèle.

**Point à arbitrer**

La cible `YieldUsingGPS_ton_ha_` écarte le Burkina Faso et la quasi-totalité du Bénin. Son maintien ou son remplacement doit être confirmé par le porteur du projet.